In [4]:
%load_ext autoreload
%autoreload 2

import sys
import time
sys.path.insert(0, '../src')

from recsys.data import load_ratings, make_positive, temporal_split, user_items
from recsys.models import PopularModel, ItemKNN, ALSModel
from recsys.metrics import evaluate, coverage, mean_popularity

ratings = load_ratings('../data')
pos = make_positive(ratings)
train, test = temporal_split(pos)
truth = user_items(test)
users = list(truth)

n_items = train['item_id'].nunique()
item_counts = train['item_id'].value_counts().to_dict()

models = {
    'Popular': PopularModel(),
    'ItemKNN': ItemKNN(),
    'ALS': ALSModel(),
}

for name, model in models.items():
    start = time.time()
    model.fit(train)
    recs = model.recommend(users, k=50)
    print(f'{name}  ({time.time() - start:.1f} сек)')
    for k in [10, 20, 50]:
        print('  ', evaluate(recs, truth, k))
    print(f'   охват@50: {coverage(recs, n_items, 50):.1%}, '
          f'средняя популярность@50: {mean_popularity(recs, item_counts, 50):.0f}')

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Popular  (0.1 сек)
   {'recall@10': 0.042648219581247405, 'ndcg@10': 0.07913936652165218}
   {'recall@20': 0.08145251724242082, 'ndcg@20': 0.08686627364132175}
   {'recall@50': 0.15499870462422113, 'ndcg@50': 0.10946566329753124}
   охват@50: 7.3%, средняя популярность@50: 1364
ItemKNN  (1.4 сек)
   {'recall@10': 0.06659914870581359, 'ndcg@10': 0.10443951602538894}
   {'recall@20': 0.11140003893622212, 'ndcg@20': 0.11384667705757714}
   {'recall@50': 0.21311636597979902, 'ndcg@50': 0.1473315026996469}
   охват@50: 23.9%, средняя популярность@50: 1192
ALS  (4.1 сек)
   {'recall@10': 0.06976206474695461, 'ndcg@10': 0.0949403315582549}
   {'recall@20': 0.1265988944575447, 'ndcg@20': 0.11237875226500757}
   {'recall@50': 0.2556262573623473, 'ndcg@50': 0.158271146360615}
   охват@50: 43.5%, средняя популярность@50: 838
